In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import os
from tqdm import tqdm

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.10.0


In [2]:
# GPU Configuration
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU detected and configured")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("⚠ No GPU detected - Running on CPU")
print("=" * 60 + "\n")

GPU CONFIGURATION
TensorFlow version: 2.10.0
Num GPUs Available: 1
✓ GPU detected and configured



## Configuration

In [3]:
# Federated Learning Configuration
NUM_CLIENTS = 50
NUM_ROUNDS = 100        
FIXED_EPOCHS = 10         # Fixed epochs for all clients each round
BATCH_SIZE = 32           # SAME as original (32 not 64)

# Dataset details
SAMPLES_PER_CLASS_PER_CLIENT = 60
SAMPLES_PER_CLIENT = 1200  # 60000 total / 50 clients

# Adaptive Training Parameters
NUM_BOTTOM_CLIENTS = 10   # Number of lowest performers to get adaptive training
MAX_ADAPTIVE_EPOCHS = 50  # Maximum total epochs (including fixed 10)
ADAPTIVE_EPOCHS_STEP = 5 # Train in steps of 5 epochs during adaptive phase

# Directories
CLIENT_DATA_DIR = 'dataset_60perclass/clients_50_from_100x60'
TEST_DATA_DIR = 'dataset_60perclass'  # Keep same 10k disjoint test set as before
RESULTS_DIR = 'results_adaptive_weight_rejection_60_per_class_50_clients'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/plots', exist_ok=True)


## Load Data

In [4]:
# Load test data (common for all clients)
print("Loading common test dataset...")
test_file = os.path.join(TEST_DATA_DIR, 'test_10000_samples_disjoint.npz')
test_data = np.load(test_file)

x_test = test_data['x'] / 255.0
y_test = test_data['y']
x_test = x_test.reshape(len(x_test), 28*28)

if len(x_test) != 10000:
    raise ValueError(f"Expected 10000 test samples, found {len(x_test)}")

print(f"✓ Test data loaded from {test_file}: {x_test.shape}")
print(f"  Labels shape: {y_test.shape}")

Loading common test dataset...
✓ Test data loaded from dataset_60perclass\test_10000_samples_disjoint.npz: (10000, 784)
  Labels shape: (10000,)


In [5]:
# Load all client data
print(f"\nLoading data for {NUM_CLIENTS} clients...")
client_data = []

for client_id in range(1, NUM_CLIENTS + 1):
    client_file = os.path.join(CLIENT_DATA_DIR, f'client_{client_id}.npz')
    data = np.load(client_file)
    
    x_client = data['x'] / 255.0
    y_client = data['y']
    x_client = x_client.reshape(len(x_client), 28*28)
    
    client_data.append({
        'x_train': x_client,
        'y_train': y_client,
        'x_test': x_test,
        'y_test': y_test
    })
    
    if client_id % 20 == 0:
        print(f"  Loaded {client_id}/{NUM_CLIENTS} clients")

print(f"\n✓ All {NUM_CLIENTS} clients loaded successfully")
print(f"  Each client: {len(client_data[0]['x_train'])} training samples")
print(f"  Common test set: {len(x_test)} samples\n")


Loading data for 50 clients...
  Loaded 20/50 clients
  Loaded 40/50 clients

✓ All 50 clients loaded successfully
  Each client: 1200 training samples
  Common test set: 10000 samples



## Model Architecture

In [6]:
# Define model - SAME as original weight rejection (simpler, proven to work)
def create_model():
    """Lightweight model optimized for small datasets"""
    model = keras.Sequential([
        keras.layers.Dense(64, input_shape=(784,), activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation="softmax")
    ])
    
    # Compile with SAME hyperparameters as original
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),  # Higher LR than before
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Test model
print("Testing model architecture...")
test_model = create_model()
test_model.summary()
print("\n✓ Model architecture validated (same as working version)\n")

Testing model architecture...
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                50240     
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense_2 (Dense)             (None, 10)                330       
                                                                 
Total params: 52,650
Trainable params: 52,650
Non-trainable params: 0
_________________________________________________________________

✓ Model architecture 

## Helper Functions

In [ ]:
# Average aggregation function
def fed_avg(weights_list):
    """Aggregate weights from all clients using coordinate-wise average."""
    avg_weights = []
    for weights_tuple in zip(*weights_list):
        avg_weights.append(np.mean(weights_tuple, axis=0))
    return avg_weights

print("✓ Average aggregation function defined")

✓ Median aggregation function defined


## Initialize Training

In [8]:
# Initialize global model and tracking
print("=" * 60)
print("INITIALIZING FEDERATED LEARNING")
print("=" * 60)

global_model = create_model()
global_weights = global_model.get_weights()

# Tracking arrays
client_test_acc_history = [[] for _ in range(NUM_CLIENTS)]
client_train_acc_history = [[] for _ in range(NUM_CLIENTS)]
client_best_weights = [None for _ in range(NUM_CLIENTS)]
client_best_test_acc = [0.0 for _ in range(NUM_CLIENTS)]
client_best_train_acc = [0.0 for _ in range(NUM_CLIENTS)]
client_rejections = [[] for _ in range(NUM_CLIENTS)]
client_epochs_used = [[] for _ in range(NUM_CLIENTS)]
client_adaptive_rounds = [[] for _ in range(NUM_CLIENTS)]

print("✓ Global model initialized")
print("✓ Tracking arrays created")
print("✓ Weight rejection + adaptive epochs ready\n")

INITIALIZING FEDERATED LEARNING
✓ Global model initialized
✓ Tracking arrays created
✓ Weight rejection + adaptive epochs ready



## Main Training Loop

In [9]:
# Main federated training loop
print("=" * 70)
print("STARTING FEDERATED TRAINING")
print("=" * 70 + "\n")

for round_num in tqdm(range(NUM_ROUNDS), desc="Communication Rounds", unit="round"):
    print(f"\n{'='*70}")
    print(f"ROUND {round_num + 1}/{NUM_ROUNDS}")
    print(f"{'='*70}")
    
    local_weights = []
    round_test_accs = []
    round_train_accs = []
    round_rejections = 0
    round_acceptances = 0
    round_client_results = []  # Store (client_id, test_acc, epochs_used) for sorting
    
    # ═══════════════════════════════════════════════════════════
    # PHASE 1: Fixed Training for ALL Clients
    # ═══════════════════════════════════════════════════════════
    print(f"\nPhase 1: Training all {NUM_CLIENTS} clients for {FIXED_EPOCHS} epochs...")
    
    client_round_data = {}  # Store results before weight rejection decision
    
    for client_id in tqdm(range(NUM_CLIENTS), desc="Fixed Training", unit="client", leave=False):
        # Create fresh model
        client_model = create_model()
        client_model.set_weights(global_weights)
        
        # Get client's data
        x_train = client_data[client_id]['x_train']
        y_train = client_data[client_id]['y_train']
        x_test = client_data[client_id]['x_test']
        y_test = client_data[client_id]['y_test']
        
        # Store old accuracy for comparison
        old_test_acc = client_best_test_acc[client_id]
        old_train_acc = client_best_train_acc[client_id]
        
        # Train for fixed epochs (SIMPLE, FAST)
        history = client_model.fit(
            x_train, y_train,
            epochs=FIXED_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )
        fixed_train_acc = float(history.history['accuracy'][-1])
        
        # Evaluate on test data
        _, new_test_acc = client_model.evaluate(x_test, y_test, verbose=0)
        new_weights = [w.copy() for w in client_model.get_weights()]
        
        # Store results temporarily
        client_round_data[client_id] = {
            'old_acc': old_test_acc,
            'old_train_acc': old_train_acc,
            'new_acc': new_test_acc,
            'new_train_acc': fixed_train_acc,
            'new_weights': new_weights,
            'epochs_used': FIXED_EPOCHS,
            'got_adaptive': False
        }
        
        # Store for sorting (to find bottom performers)
        round_client_results.append((client_id, new_test_acc, FIXED_EPOCHS))
        
        # Clear GPU memory after each client training
        del client_model
        tf.keras.backend.clear_session()
    
    # ═══════════════════════════════════════════════════════════
    # PHASE 2: Adaptive Training for Bottom Performers
    # ═══════════════════════════════════════════════════════════
    print(f"\nPhase 2: Adaptive training for bottom {NUM_BOTTOM_CLIENTS} clients...")
    
    # Sort by test accuracy (ascending - lowest first)
    round_client_results.sort(key=lambda x: x[1])
    bottom_clients = [cid for cid, _, _ in round_client_results[:NUM_BOTTOM_CLIENTS]]
    
    print(f"Bottom {NUM_BOTTOM_CLIENTS} clients identified:")
    for i, (cid, acc, _) in enumerate(round_client_results[:NUM_BOTTOM_CLIENTS]):
        print(f"  {i+1}. Client {cid+1}: {acc*100:.2f}%")
    
    adaptive_improved = 0
    for client_id in tqdm(bottom_clients, desc="Adaptive Training", unit="client", leave=False):
        data = client_round_data[client_id]
        target_acc = data['old_acc']
        current_acc = data['new_acc']
        
        # Only do adaptive training if they haven't improved yet
        if current_acc <= target_acc and target_acc > 0:
            # Create model with current weights
            client_model = create_model()
            client_model.set_weights(data['new_weights'])
            
            x_train = client_data[client_id]['x_train']
            y_train = client_data[client_id]['y_train']
            x_test = client_data[client_id]['x_test']
            y_test = client_data[client_id]['y_test']
            
            best_adaptive_acc = current_acc
            best_adaptive_train_acc = data['new_train_acc']
            best_adaptive_weights = data['new_weights']
            total_epochs = FIXED_EPOCHS
            
            # Train in steps until improvement or max epochs
            while total_epochs < MAX_ADAPTIVE_EPOCHS:
                client_model.fit(
                    x_train, y_train,
                    epochs=ADAPTIVE_EPOCHS_STEP,
                    batch_size=BATCH_SIZE,
                    verbose=0
                )
                
                total_epochs += ADAPTIVE_EPOCHS_STEP
                _, test_acc = client_model.evaluate(x_test, y_test, verbose=0)
                _, train_acc = client_model.evaluate(x_train, y_train, verbose=0)
                
                # Track best
                if test_acc > best_adaptive_acc:
                    best_adaptive_acc = test_acc
                    best_adaptive_train_acc = train_acc
                    best_adaptive_weights = [w.copy() for w in client_model.get_weights()]
                
                # Stop if improved beyond target
                if test_acc > target_acc:
                    adaptive_improved += 1
                    break
            
            # Update client data with best adaptive results
            client_round_data[client_id]['new_acc'] = best_adaptive_acc
            client_round_data[client_id]['new_train_acc'] = best_adaptive_train_acc
            client_round_data[client_id]['new_weights'] = best_adaptive_weights
            client_round_data[client_id]['epochs_used'] = total_epochs
            client_round_data[client_id]['got_adaptive'] = True
            
            # Clear GPU memory after adaptive training
            del client_model
            tf.keras.backend.clear_session()
    
    if adaptive_improved > 0:
        print(f"  ✓ {adaptive_improved}/{len(bottom_clients)} clients improved with adaptive training")
    
    # ═══════════════════════════════════════════════════════════
    # PHASE 3: Weight Rejection Decision
    # ═══════════════════════════════════════════════════════════
    print(f"\nPhase 3: Weight rejection decision...")
    
    for client_id in range(NUM_CLIENTS):
        data = client_round_data[client_id]
        old_acc = data['old_acc']
        new_acc = data['new_acc']
        old_train_acc = data['old_train_acc']
        new_train_acc = data['new_train_acc']
        epochs_used = data['epochs_used']
        
        # DECISION: Accept only if improved
        if new_acc > old_acc:
            # ACCEPT
            client_best_weights[client_id] = data['new_weights']
            client_best_test_acc[client_id] = new_acc
            client_best_train_acc[client_id] = new_train_acc
            client_rejections[client_id].append(0)
            round_acceptances += 1
            final_test_acc = new_acc
            final_train_acc = new_train_acc
        else:
            # REJECT - keep old weights
            if client_best_weights[client_id] is None:
                # First round - accept anyway
                client_best_weights[client_id] = data['new_weights']
                client_best_test_acc[client_id] = new_acc
                client_best_train_acc[client_id] = new_train_acc
                client_rejections[client_id].append(0)
                round_acceptances += 1
                final_test_acc = new_acc
                final_train_acc = new_train_acc
            else:
                # Reject, use previous best
                client_rejections[client_id].append(1)
                round_rejections += 1
                final_test_acc = client_best_test_acc[client_id]
                final_train_acc = client_best_train_acc[client_id]
        
        # Track results
        client_test_acc_history[client_id].append(final_test_acc)
        client_train_acc_history[client_id].append(final_train_acc)
        client_epochs_used[client_id].append(epochs_used)
        client_adaptive_rounds[client_id].append(1 if data['got_adaptive'] else 0)
        round_test_accs.append(final_test_acc)
        round_train_accs.append(final_train_acc)
        
        # Collect best weights for FedAvg aggregation
        local_weights.append(client_best_weights[client_id])
    
    # ═══════════════════════════════════════════════════════════
    # FedAvg
    # ═══════════════════════════════════════════════════════════
    global_weights = fed_avg(local_weights)
    global_model.set_weights(global_weights)
    
    # Round summary
    median_test_acc = np.median(round_test_accs) * 100
    median_train_acc = np.median(round_train_accs) * 100
    min_test_acc = np.min(round_test_accs) * 100
    max_test_acc = np.max(round_test_accs) * 100
    std_test_acc = np.std([acc * 100 for acc in round_test_accs])
    total_adaptive = sum([1 for cid in range(NUM_CLIENTS) if client_round_data[cid]['got_adaptive']])
    
    print(f"\n📊 Round {round_num + 1} Summary:")
    print(f"   Median Test Accuracy: {median_test_acc:.2f}%")
    print(f"   Median Train Accuracy: {median_train_acc:.2f}%")
    print(f"   Std Dev: {std_test_acc:.2f}%")
    print(f"   Range: [{min_test_acc:.2f}%, {max_test_acc:.2f}%]")
    print(f"   Weights Accepted: {round_acceptances}/{NUM_CLIENTS}")
    print(f"   Weights Rejected: {round_rejections}/{NUM_CLIENTS}")
    print(f"   Adaptive Training: {total_adaptive}/{NUM_CLIENTS} clients")

print("\n" + "="*70)
print("FEDERATED TRAINING COMPLETE!")
print("="*70 + "\n")

STARTING FEDERATED TRAINING



Communication Rounds:   0%|          | 0/100 [00:00<?, ?round/s]


ROUND 1/100

Phase 1: Training all 50 clients for 10 epochs...



Phase 2: Adaptive training for bottom 10 clients...
Bottom 10 clients identified:
  1. Client 9: 88.77%
  2. Client 36: 88.77%
  3. Client 2: 88.85%
  4. Client 49: 88.90%
  5. Client 46: 88.93%
  6. Client 13: 88.94%
  7. Client 43: 88.98%
  8. Client 19: 89.09%
  9. Client 45: 89.09%
  10. Client 22: 89.13%


Communication Rounds:   1%|          | 1/100 [05:02<8:19:06, 302.49s/round]

...

Communication Rounds:  99%|█████████▉| 99/100 [3:44:29<02:14, 134.90s/round]


Phase 3: Weight rejection decision...

📊 Round 99 Summary:
   Median Test Accuracy: 96.01%
   Median Train Accuracy: 97.83%
   Std Dev: 0.12%
   Range: [95.71%, 96.30%]
   Weights Accepted: 3/50
   Weights Rejected: 47/50
   Adaptive Training: 10/50 clients

ROUND 100/100

Phase 1: Training all 50 clients for 10 epochs...



Phase 2: Adaptive training for bottom 10 clients...
Bottom 10 clients identified:
  1. Client 16: 95.10%
  2. Client 23: 95.23%
  3. Client 44: 95.30%
  4. Client 43: 95.34%
  5. Client 35: 95.36%
  6. Client 8: 95.41%
  7. Client 9: 95.42%
  8. Client 49: 95.50%
  9. Client 4: 95.56%
  10. Client 7: 95.56%


Communication Rounds: 100%|██████████| 100/100 [3:46:47<00:00, 136.08s/round]


Phase 3: Weight rejection decision...

📊 Round 100 Summary:
   Median Test Accuracy: 96.01%
   Median Train Accuracy: 97.88%
   Std Dev: 0.12%
   Range: [95.71%, 96.30%]
   Weights Accepted: 3/50
   Weights Rejected: 47/50
   Adaptive Training: 10/50 clients

FEDERATED TRAINING COMPLETE!



## Results Analysis

In [10]:
# Calculate final statistics
final_test_accs = [client_test_acc_history[i][-1] * 100 for i in range(NUM_CLIENTS)]
final_train_accs = [client_train_acc_history[i][-1] * 100 for i in range(NUM_CLIENTS)]
total_rejections_per_client = [sum(client_rejections[i]) for i in range(NUM_CLIENTS)]
total_epochs_per_client = [sum(client_epochs_used[i]) for i in range(NUM_CLIENTS)]
total_adaptive_rounds = [sum(client_adaptive_rounds[i]) for i in range(NUM_CLIENTS)]

print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(f"Median Final Test Accuracy: {np.median(final_test_accs):.2f}%")
print(f"Median Final Train Accuracy: {np.median(final_train_accs):.2f}%")
print(f"Std Dev (Test): {np.std(final_test_accs):.2f}%")
print(f"Std Dev (Train): {np.std(final_train_accs):.2f}%")
print(f"Test Accuracy Range: [{np.min(final_test_accs):.2f}%, {np.max(final_test_accs):.2f}%]")
print(f"Train Accuracy Range: [{np.min(final_train_accs):.2f}%, {np.max(final_train_accs):.2f}%]")
print()
print("Weight Rejection Statistics:")
print(f"  Total Rejections: {sum(total_rejections_per_client)} / {NUM_CLIENTS * NUM_ROUNDS}")
print(f"  Rejection Rate: {sum(total_rejections_per_client) / (NUM_CLIENTS * NUM_ROUNDS) * 100:.2f}%")
print(f"  Median Rejections per Client: {np.median(total_rejections_per_client):.2f}")
print()
print("Adaptive Training Statistics:")
print(f"  Total Adaptive Rounds: {sum(total_adaptive_rounds)}")
print(f"  Median Adaptive Rounds per Client: {np.median(total_adaptive_rounds):.2f}")
print(f"  Total Epochs Used: {sum(total_epochs_per_client)}")
print(f"  Median Epochs per Client: {np.median(total_epochs_per_client):.1f}")
print("=" * 70 + "\n")

FINAL RESULTS
Median Final Test Accuracy: 96.01%
Median Final Train Accuracy: 97.88%
Std Dev (Test): 0.12%
Std Dev (Train): 0.48%
Test Accuracy Range: [95.71%, 96.30%]
Train Accuracy Range: [96.83%, 98.75%]

Weight Rejection Statistics:
  Total Rejections: 3963 / 5000
  Rejection Rate: 79.26%
  Median Rejections per Client: 80.00

Adaptive Training Statistics:
  Total Adaptive Rounds: 913
  Median Adaptive Rounds per Client: 16.00
  Total Epochs Used: 85850
  Median Epochs per Client: 1622.5



In [11]:
# Detailed per-client results
print("PER-CLIENT RESULTS:")
print("-" * 128)
print(f"{'Client':<8} {'Final Test':<12} {'Final Train':<12} {'Total Epochs':<14} {'Rejections':<14} {'Adaptive Rounds':<16}")
print("-" * 128)

for client_id in range(NUM_CLIENTS):
    final_test = final_test_accs[client_id]
    final_train = final_train_accs[client_id]
    total_epochs = total_epochs_per_client[client_id]
    rejections = total_rejections_per_client[client_id]
    adaptive = total_adaptive_rounds[client_id]
    print(f"{client_id + 1:<8} {final_test:.2f}%{'':<6} {final_train:.2f}%{'':<6} {total_epochs:<14} {rejections:<14} {adaptive:<16}")

print("-" * 128 + "\n")

PER-CLIENT RESULTS:
--------------------------------------------------------------------------------------------------------------------------------
Client   Final Test   Final Train  Total Epochs   Rejections     Adaptive Rounds 
--------------------------------------------------------------------------------------------------------------------------------
1        96.05%       96.92%       1120           80             3               
2        96.08%       97.92%       1125           78             4               
3        95.99%       98.42%       1840           80             21              
4        95.88%       98.00%       2650           81             42              
5        95.88%       97.25%       1925           81             24              
6        96.10%       97.50%       1360           78             9               
7        95.95%       98.25%       2765           80             45              
8        96.02%       97.42%       2055           78             2

## Round-wise Accuracy Tables

In [12]:
# Create client x round tables for test and training accuracy
TARGET_ROUNDS = 100
available_rounds = len(client_test_acc_history[0])
num_table_rounds = min(TARGET_ROUNDS, available_rounds)

client_labels = [f"Client_{i+1}" for i in range(NUM_CLIENTS)]
round_labels = [f"Round_{r+1}" for r in range(num_table_rounds)]

test_matrix = np.array([hist[:num_table_rounds] for hist in client_test_acc_history]) * 100
train_matrix = np.array([hist[:num_table_rounds] for hist in client_train_acc_history]) * 100

test_acc_table = pd.DataFrame(test_matrix, index=client_labels, columns=round_labels)
train_acc_table = pd.DataFrame(train_matrix, index=client_labels, columns=round_labels)

# Save as CSV for full inspection in spreadsheet tools
test_csv_path = f'{RESULTS_DIR}/test_accuracy_client_round_table_{num_table_rounds}_rounds.csv'
train_csv_path = f'{RESULTS_DIR}/train_accuracy_client_round_table_{num_table_rounds}_rounds.csv'
test_acc_table.to_csv(test_csv_path, index=True)
train_acc_table.to_csv(train_csv_path, index=True)

print("=" * 70)
print("CLIENT x ROUND ACCURACY TABLES")
print("=" * 70)
print(f"Rows (clients): {test_acc_table.shape[0]}")
print(f"Columns (rounds): {test_acc_table.shape[1]}")
print(f"Target rounds requested: {TARGET_ROUNDS}")
print(f"Rounds available from current run: {available_rounds}")
print(f"Using rounds in table: {num_table_rounds}")
print()
print(f"✓ Test accuracy table saved: {test_csv_path}")
print(f"✓ Train accuracy table saved: {train_csv_path}")
print()
print("Test accuracy table (first 5 clients, first 10 rounds):")
display(test_acc_table.iloc[:5, :10])
print("\nTrain accuracy table (first 5 clients, first 10 rounds):")
display(train_acc_table.iloc[:5, :10])
print("=" * 70)

CLIENT x ROUND ACCURACY TABLES
Rows (clients): 50
Columns (rounds): 100
Target rounds requested: 100
Rounds available from current run: 100
Using rounds in table: 100

✓ Test accuracy table saved: results_adaptive_weight_rejection_60_per_class_50_clients/test_accuracy_client_round_table_100_rounds.csv
✓ Train accuracy table saved: results_adaptive_weight_rejection_60_per_class_50_clients/train_accuracy_client_round_table_100_rounds.csv

Test accuracy table (first 5 clients, first 10 rounds):


,Round_1,Round_2,Round_3,Round_4,Round_5,Round_6,Round_7,Round_8,Round_9,Round_10
Client_1,89.980000,91.360003,92.490000,93.750000,93.800002,94.389999,94.809997,94.970000,94.970000,95.209998
Client_2,88.849998,90.549999,92.320001,93.260002,93.400002,94.169998,94.550002,94.639999,94.690001,94.959998
Client_3,89.870000,91.500002,92.339998,93.169999,93.279999,93.930000,94.510001,94.510001,94.630003,94.630003
Client_4,89.440000,91.610003,92.479998,93.400002,93.800002,94.349998,94.349998,94.410002,94.739997,94.739997
Client_5,90.120000,91.360003,92.610002,93.580002,93.660003,94.010001,94.389999,94.410002,94.819999,94.819999



Train accuracy table (first 5 clients, first 10 rounds):


,Round_1,Round_2,Round_3,Round_4,Round_5,Round_6,Round_7,Round_8,Round_9,Round_10
Client_1,86.416668,91.666669,93.750000,94.999999,94.749999,96.416664,95.583332,95.916665,95.916665,97.333336
Client_2,87.666667,92.166668,94.499999,95.166665,95.166665,95.999998,97.250003,96.499997,99.833333,97.166669
Client_3,86.083335,91.750002,94.333333,95.916665,95.333332,96.749997,97.166669,97.166669,97.416669,97.416669
Client_4,87.916666,92.333335,93.416667,95.166665,95.666665,95.833331,95.833331,97.916669,98.083335,98.083335
Client_5,86.333334,92.083335,93.250000,94.583333,95.416665,96.166664,96.499997,95.916665,96.499997,96.499997


## Visualization

In [13]:
# Plot 1: Median accuracy across all clients
print("Creating median accuracy plot...")

plt.figure(figsize=(12, 7))
rounds = range(1, NUM_ROUNDS + 1)

median_accs = []
std_accs = []
for r in range(NUM_ROUNDS):
    round_accs = [client_test_acc_history[i][r] * 100 for i in range(NUM_CLIENTS)]
    median_accs.append(np.median(round_accs))
    std_accs.append(np.std(round_accs))

plt.plot(rounds, median_accs, marker='o', linewidth=3, markersize=8, 
         color='#2E86AB', label='Median Test Accuracy')
plt.fill_between(rounds, 
                 np.array(median_accs) - np.array(std_accs),
                 np.array(median_accs) + np.array(std_accs),
                 alpha=0.3, color='#A8DADC', label='±1 Std Dev')

plt.title('Median Test Accuracy - Adaptive Epochs + Weight Rejection', 
          fontsize=16, fontweight='bold')
plt.xlabel('Communication Round', fontsize=13)
plt.ylabel('Test Accuracy (%)', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([50, 100])

plt.savefig(f'{RESULTS_DIR}/plots/median_accuracy.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: median_accuracy.png")
plt.close()

Creating median accuracy plot...
✓ Saved: median_accuracy.png


In [14]:
# Plot 2: Individual client accuracy curves (dynamic grid)
print("Creating individual client accuracy plots (dynamic grid)...")

import math
grid_cols = int(math.ceil(math.sqrt(NUM_CLIENTS)))
grid_rows = int(math.ceil(NUM_CLIENTS / grid_cols))

fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(grid_cols * 3, grid_rows * 3))
fig.suptitle('Test Accuracy per Client - Adaptive Epochs + Weight Rejection', 
             fontsize=20, fontweight='bold', y=0.995)

# Normalize axes to a 2D array for consistent indexing
axes = np.array(axes).reshape(grid_rows, grid_cols)

rounds = range(1, NUM_ROUNDS + 1)

for client_id in range(NUM_CLIENTS):
    row = client_id // grid_cols
    col = client_id % grid_cols
    ax = axes[row, col]
    
    accs = [acc * 100 for acc in client_test_acc_history[client_id]]
    final_acc = accs[-1]
    improvement = final_acc - accs[0]
    rejections = total_rejections_per_client[client_id]
    adaptive = total_adaptive_rounds[client_id]
    
    color = '#2E86AB' if improvement >= 0 else '#E63946'
    ax.plot(rounds, accs, marker='o', linewidth=2, markersize=4, color=color)
    ax.fill_between(rounds, accs, alpha=0.3, color=color)
    
    title_color = 'green' if improvement >= 0 else 'red'
    ax.set_title(f'C{client_id+1} ({improvement:+.1f}%) R:{rejections} A:{adaptive}', 
                fontsize=9, fontweight='bold', color=title_color)
    
    ax.text(NUM_ROUNDS, final_acc, f'{final_acc:.0f}%', 
            fontsize=8, fontweight='bold', verticalalignment='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim(0, 100)
    ax.set_xlim(0.5, NUM_ROUNDS + 0.5)
    
    if row == grid_rows - 1:
        ax.set_xlabel('Round', fontsize=8)
    if col == 0:
        ax.set_ylabel('Accuracy (%)', fontsize=8)

# Hide any unused subplot axes
for idx in range(NUM_CLIENTS, grid_rows * grid_cols):
    row = idx // grid_cols
    col = idx % grid_cols
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/plots/all_clients_grid.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: all_clients_grid.png")
plt.close()

Creating individual client accuracy plots (dynamic grid)...
✓ Saved: all_clients_grid.png


In [15]:
# Plot 3: Rejection heatmap
print("Creating weight rejection heatmap...")

plt.figure(figsize=(14, 10))

rejection_matrix = np.zeros((NUM_CLIENTS, NUM_ROUNDS))
for i in range(NUM_CLIENTS):
    for j in range(NUM_ROUNDS):
        rejection_matrix[i, j] = client_rejections[i][j]

plt.imshow(rejection_matrix, cmap='RdYlGn_r', aspect='auto', interpolation='nearest')
plt.colorbar(label='Rejected (1=Yes, 0=No)')
plt.title('Weight Rejection Status per Client per Round', fontsize=16, fontweight='bold')
plt.xlabel('Communication Round', fontsize=13)
plt.ylabel('Client ID', fontsize=13)
plt.tight_layout()

plt.savefig(f'{RESULTS_DIR}/plots/rejection_heatmap.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: rejection_heatmap.png")
plt.close()

Creating weight rejection heatmap...
✓ Saved: rejection_heatmap.png


In [16]:
# Plot 4: Epochs distribution
print("Creating epochs distribution plot...")

plt.figure(figsize=(12, 7))

epochs_matrix = []
for r in range(NUM_ROUNDS):
    round_epochs = [client_epochs_used[i][r] for i in range(NUM_CLIENTS)]
    epochs_matrix.append(round_epochs)

bp = plt.boxplot(epochs_matrix, positions=range(1, NUM_ROUNDS + 1), 
                 widths=0.6, patch_artist=True,
                 boxprops=dict(facecolor='lightblue', color='blue'),
                 medianprops=dict(color='red', linewidth=2))

plt.title('Epochs Distribution per Round (Fixed + Adaptive)', 
          fontsize=16, fontweight='bold')
plt.xlabel('Communication Round', fontsize=13)
plt.ylabel('Epochs Used', fontsize=13)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()

plt.savefig(f'{RESULTS_DIR}/plots/epochs_distribution.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: epochs_distribution.png")
plt.close()

print("\n✅ All plots generated!\n")

Creating epochs distribution plot...
✓ Saved: epochs_distribution.png

✅ All plots generated!



## Save Results

In [17]:
# Save results and model
print("Saving results...")

results_data = {
    'client_test_acc_history': client_test_acc_history,
    'client_train_acc_history': client_train_acc_history,
    'client_rejections': client_rejections,
    'client_epochs_used': client_epochs_used,
    'client_adaptive_rounds': client_adaptive_rounds,
    'final_test_accs': final_test_accs,
    'final_train_accs': final_train_accs,
    'config': {
        'num_clients': NUM_CLIENTS,
        'num_rounds': NUM_ROUNDS,
        'fixed_epochs': FIXED_EPOCHS,
        'max_adaptive_epochs': MAX_ADAPTIVE_EPOCHS,
        'num_bottom_clients': NUM_BOTTOM_CLIENTS,
        'batch_size': BATCH_SIZE
    }
}

np.save(f'{RESULTS_DIR}/training_results.npy', results_data, allow_pickle=True)
print(f"✓ Results saved: {RESULTS_DIR}/training_results.npy")

# Save global model
global_model.save(f'{RESULTS_DIR}/global_model.h5')
print(f"✓ Model saved: {RESULTS_DIR}/global_model.h5")

print("\n✅ COMPLETE! All results saved to:", RESULTS_DIR)

Saving results...
✓ Results saved: results_adaptive_weight_rejection_60_per_class_50_clients/training_results.npy
✓ Model saved: results_adaptive_weight_rejection_60_per_class_50_clients/global_model.h5

✅ COMPLETE! All results saved to: results_adaptive_weight_rejection_60_per_class_50_clients
